# 🚀 TRẢI NGHIỆM & THỬ NGHIỆM MÔ HÌNH FINE-TUNED QWEN2-VL TRÊN GPU TESLA T4
## Hệ thống Document Visual Question Answering (DocVQA) Hóa đơn Tiếng Việt

In [ ]:
# 1. CÀI ĐẶT THƯ VIỆN
print("📦 Đang cài đặt thư viện cho GPU...")
!pip uninstall -y -q torchao
!pip install -q --no-deps qwen-vl-utils==0.0.8
!pip install -q "transformers==4.46.2" "peft==0.13.2" "accelerate==0.34.2" gradio pillow torchvision

import os
import sys
import gc
import time
import json
import zipfile
from pathlib import Path

import torch
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from peft import PeftModel
from qwen_vl_utils import process_vision_info

print(f"🔥 Đang sử dụng GPU: {torch.cuda.get_device_name(0)}")
print(f"🧠 Tổng VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


In [ ]:
# 2. NẠP MODEL VÀ LORA ADAPTER ĐÃ TRAIN
adapter_dir = None
for root, dirs, files in os.walk("/kaggle"):
    if "adapter_model.safetensors" in files:
        adapter_dir = root
        print(f"✅ Tìm thấy LoRA Adapter tại: {adapter_dir}")
        break
    for f in files:
        if f.endswith(".zip") and "lora" in f.lower():
            extract_to = "/kaggle/working/loaded_lora"
            os.makedirs(extract_to, exist_ok=True)
            with zipfile.ZipFile(os.path.join(root, f), 'r') as zf:
                zf.extractall(extract_to)
            adapter_dir = os.path.join(extract_to, "qwen2_vl_lora_adapters") if os.path.exists(os.path.join(extract_to, "qwen2_vl_lora_adapters")) else extract_to
            print(f"✅ Đã giải nén LoRA Adapter vào: {adapter_dir}")
            break
    if adapter_dir:
        break

model_id = "Qwen/Qwen2-VL-2B-Instruct"
print(f"🧠 Đang nạp Base Model: {model_id} (FP16)... ")
processor = AutoProcessor.from_pretrained(model_id, min_pixels=256*28*28, max_pixels=768*28*28)
base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

if adapter_dir and os.path.exists(adapter_dir):
    print(f"🎯 Đang kích hoạt LoRA Adapter từ {adapter_dir}...")
    model = PeftModel.from_pretrained(base_model, adapter_dir)
    print("🎉 KÍCH HOẠT LORA ADAPTER THÀNH CÔNG! Mô hình đã sẵn sàng suy luận!")
else:
    model = base_model

model.eval()


In [ ]:
# 3. HÀM SUY LUẬN BÓC TÁCH THÔNG TIN HÓA ĐƠN
def ask_receipt(image_input, question: str):
    if isinstance(image_input, str):
        image = Image.open(image_input).convert("RGB")
    else:
        image = image_input.convert("RGB")
        
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": question}
            ]
        }
    ]
    
    t0 = time.time()
    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[prompt],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to("cuda")
    
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=96,
            do_sample=False
        )
        trimmed_generated_ids = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = processor.batch_decode(
            trimmed_generated_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )[0].strip()
        
    latency = time.time() - t0
    return output_text, latency

print("✅ Hàm ask_receipt() đã sẵn sàng!")


In [ ]:
# 4. KHỞI CHẠY WEB APP GRADIO PUBLIC VÀ GIỮ KẾT NỐI SỐNG LIÊN TỤC TRÊN GPU T4
import gradio as gr

def gradio_interface(image, question):
    if image is None:
        return "Vui lòng tải lên một hình ảnh hóa đơn.", "0.00s"
    if not question or not question.strip():
        return "Vui lòng nhập câu hỏi cần bóc tách.", "0.00s"
    ans, lat = ask_receipt(image, question)
    return ans, f"{lat:.2f} giây (GPU Tesla T4)"

with gr.Blocks(title="Document VQA - Hệ Thống Bóc Tách Hóa Đơn") as demo:
    gr.Markdown("# 🧾 Document VQA Live Demo: Qwen2-VL-2B + LoRA (Fine-Tuned)")
    gr.Markdown("**Tải lên ảnh hóa đơn bất kỳ của bạn** và bấm **Bóc Tách Thông Tin** để nhận kết quả siêu tốc trên GPU Tesla T4!")
    
    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(type="pil", label="Ảnh hóa đơn của bạn")
            question_input = gr.Textbox(
                label="Câu hỏi",
                placeholder="Ví dụ: Tổng tiền thanh toán là bao nhiêu?",
                value="Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?"
            )
            btn = gr.Button("🔍 Bóc Tách Thông Tin", variant="primary")
            
            gr.Markdown("### 💡 Gợi ý câu hỏi mẫu:")
            gr.Examples(
                examples=[
                    ["Tên đơn vị / người bán hàng trên hóa đơn là gì?"],
                    ["Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?"],
                    ["Ngày giờ lập hóa đơn là khi nào?"],
                    ["Địa chỉ của đơn vị bán hàng là ở đâu?"],
                    ["Mã số thuế của đơn vị bán hàng là gì?"]
                ],
                inputs=[question_input]
            )
            
        with gr.Column(scale=1):
            answer_output = gr.Textbox(label="Kết quả bóc tách từ Mô hình (Entity Value)", lines=4)
            latency_output = gr.Textbox(label="Độ trễ suy luận")
            
    btn.click(fn=gradio_interface, inputs=[image_input, question_input], outputs=[answer_output, latency_output])

print("🌐 Đang khởi chạy giao diện Gradio Live Web Server...")
demo.queue()
app, local_url, share_url = demo.launch(share=True, show_error=True)
print(f"\n🔥 PUBLIC WEB APP URL ĐANG HOẠT ĐỘNG: {share_url}")

# Giữ kết nối Web sống liên tục trên GPU
for minute in range(120):
    time.sleep(60)
    if minute % 10 == 0:
        print(f"[{time.strftime('%H:%M:%S')}] Server đang chạy trực tiếp trên GPU: {share_url}")
